In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import lazypredict
from lazypredict.Supervised import LazyRegressor
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')



In [12]:

# Veri işleme ve model eğitme fonksiyonu
def prepare_model():
    print("Veri seti yükleniyor ve işleniyor...")
    # Veriyi yükle
    df = pd.read_csv('dataset/train.csv')

    # Veri ön izleme
    print("\nVeri Seti Önizleme:")
    print(df.head())

    # Özellik mühendisliği
    df['engine_power'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
    df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+) Cylinder').astype(float)
    df['fuel_type'] = df['fuel_type'].str.replace('Gasoline/Mild Electric Hybrid', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Plug-In Electric/Gas', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Gas/Electric Hybrid', 'Hybrid')
    df['age'] = 2023 - df['model_year']
    df['accident'] = df['accident'].apply(lambda x: 1 if 'accident' in str(x) else 0)
    df['clean_title'] = df['clean_title'].apply(lambda x: 1 if str(x) == 'Yes' else 0)
    df['transmission_type'] = df['transmission'].apply(
        lambda x: 'Automatic' if 'A/T' in str(x) else 'Manual' if 'M/T' in str(x) else 'Other')

    # Temizlik
    df = df.dropna(subset=['engine_power', 'engine_volume', 'cylinders'])
    df['clean_title'].fillna(0, inplace=True)

    # Kategorik ve sayısal sütunlar
    categorical_cols = ['brand', 'fuel_type', 'transmission_type']
    numerical_cols = ['model_year', 'milage', 'engine_power', 'engine_volume',
                     'cylinders', 'age', 'accident', 'clean_title']

    # Eğitim verisi
    X = df[numerical_cols + categorical_cols]
    y = df['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nLazyPredict modelleri çalıştırılıyor...")

    # LazyPredict ile regresyon modellerini karşılaştırma
    regressor = LazyRegressor()
    models, predictions = regressor.fit(X_train, X_test, y_train, y_test)

    print("\nLazyPredict Sonuçları:")
    print(models)

    # En iyi model seçimi (en düşük MAE'yi seçmek için)
    best_model = models.sort_values('MAE').iloc[0]
    print(f"\nEn iyi model: {best_model.name} (MAE: {best_model.MAE:.2f})")

    # En iyi modeli yeniden eğitme
    model = best_model.model
    model.fit(X_train, y_train)

    return model


In [13]:

# Modeli kaydet veya yükle
def get_model():
    model_file = 'car_price_model.pkl'
    if os.path.exists(model_file):
        print("\nÖnceden eğitilmiş model yükleniyor...")
        return joblib.load(model_file)
    else:
        print("\nYeni model eğitiliyor...")
        model = prepare_model()
        joblib.dump(model, model_file)
        print(f"Model '{model_file}' olarak kaydedildi.")
        return model



In [14]:
# Kullanıcı girişi alma
def get_user_input():
    print("\n" + "="*50)
    print("Araç Özelliklerini Girin:")
    print("="*50)

    brand = input("\nMarka (Örnek: Toyota, BMW, Ford): ").strip().title()
    model_year = int(input("Model Yılı (Örnek: 2015): "))
    milage = int(input("Kilometre (Örnek: 50000): "))
    fuel_type = input("Yakıt Türü (Gasoline, Diesel, Hybrid, Electric): ").strip().title()
    engine_power = float(input("Motor Gücü (HP) (Örnek: 150): "))
    engine_volume = float(input("Motor Hacmi (L) (Örnek: 2.0): "))
    cylinders = int(input("Silindir Sayısı (Örnek: 4): "))
    transmission_type = input("Şanzıman Türü (Automatic, Manual, Other): ").strip().title()
    accident = input("Kaza Geçmişi Var mı? (Evet/Hayır): ").lower() == 'evet'
    clean_title = input("Temiz Başlık? (Evet/Hayır): ").lower() == 'evet'

    # Yaş hesapla
    current_year = pd.Timestamp.now().year
    age = current_year - model_year

    # Veri sözlüğü oluştur
    input_data = {
        'brand': [brand],
        'model_year': [model_year],
        'milage': [milage],
        'fuel_type': [fuel_type],
        'engine_power': [engine_power],
        'engine_volume': [engine_volume],
        'cylinders': [cylinders],
        'transmission_type': [transmission_type],
        'age': [age],
        'accident': [1 if accident else 0],
        'clean_title': [1 if clean_title else 0]
    }

    return pd.DataFrame(input_data)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import lazypredict
from lazypredict.Supervised import LazyRegressor
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

# Veri işleme ve model eğitme fonksiyonu
def prepare_model():
    print("Veri seti yükleniyor ve işleniyor...")
    # Veriyi yükle
    df = pd.read_csv('dataset/train.csv')

    # Veri ön izleme
    print("\nVeri Seti Önizleme:")
    print(df.head())

    # Özellik mühendisliği
    df['engine_power'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
    df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+) Cylinder').astype(float)
    df['fuel_type'] = df['fuel_type'].str.replace('Gasoline/Mild Electric Hybrid', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Plug-In Electric/Gas', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Gas/Electric Hybrid', 'Hybrid')
    df['age'] = 2023 - df['model_year']
    df['accident'] = df['accident'].apply(lambda x: 1 if 'accident' in str(x) else 0)
    df['clean_title'] = df['clean_title'].apply(lambda x: 1 if str(x) == 'Yes' else 0)
    df['transmission_type'] = df['transmission'].apply(
        lambda x: 'Automatic' if 'A/T' in str(x) else 'Manual' if 'M/T' in str(x) else 'Other')

    # Temizlik
    df = df.dropna(subset=['engine_power', 'engine_volume', 'cylinders'])
    df['clean_title'].fillna(0, inplace=True)

    # Kategorik ve sayısal sütunlar
    categorical_cols = ['brand', 'fuel_type', 'transmission_type']
    numerical_cols = ['model_year', 'milage', 'engine_power', 'engine_volume',
                     'cylinders', 'age', 'accident', 'clean_title']

    # Eğitim verisi
    X = df[numerical_cols + categorical_cols]
    y = df['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nLazyPredict modelleri çalıştırılıyor...")

    # LazyPredict ile regresyon modellerini karşılaştırma
    regressor = LazyRegressor()
    models, predictions = regressor.fit(X_train, X_test, y_train, y_test)

    print("\nLazyPredict Sonuçları:")
    print(models)

    # En iyi model seçimi (en düşük MAE'yi seçmek için)
    best_model = models.sort_values('MAE').iloc[0]
    print(f"\nEn iyi model: {best_model.name} (MAE: {best_model.MAE:.2f})")

    # En iyi modeli yeniden eğitme
    model = best_model.model
    model.fit(X_train, y_train)

    # Modeli kaydet
    joblib.dump(model, 'best_car_price_model.pkl')
    print("\nEn iyi model kaydedildi: best_car_price_model.pkl")

    return model

# Modeli kaydet veya yükle
def get_model():
    model_file = 'best_car_price_model.pkl'
    if os.path.exists(model_file):
        print("\nÖnceden eğitilmiş model yükleniyor...")
        return joblib.load(model_file)
    else:
        print("\nYeni model eğitiliyor...")
        model = prepare_model()
        return model

# Kullanıcı girişi alma
def get_user_input():
    print("\n" + "="*50)
    print("Araç Özelliklerini Girin:")
    print("="*50)

    brand = input("\nMarka (Örnek: Toyota, BMW, Ford): ").strip().title()
    model_year = int(input("Model Yılı (Örnek: 2015): "))
    milage = int(input("Kilometre (Örnek: 50000): "))
    fuel_type = input("Yakıt Türü (Gasoline, Diesel, Hybrid, Electric): ").strip().title()
    engine_power = float(input("Motor Gücü (HP) (Örnek: 150): "))
    engine_volume = float(input("Motor Hacmi (L) (Örnek: 2.0): "))
    cylinders = int(input("Silindir Sayısı (Örnek: 4): "))
    transmission_type = input("Şanzıman Türü (Automatic, Manual, Other): ").strip().title()
    accident = input("Kaza Geçmişi Var mı? (Evet/Hayır): ").lower() == 'evet'
    clean_title = input("Temiz Başlık? (Evet/Hayır): ").lower() == 'evet'

    # Yaş hesapla
    current_year = pd.Timestamp.now().year
    age = current_year - model_year

    # Veri sözlüğü oluştur
    input_data = {
        'brand': [brand],
        'model_year': [model_year],
        'milage': [milage],
        'fuel_type': [fuel_type],
        'engine_power': [engine_power],
        'engine_volume': [engine_volume],
        'cylinders': [cylinders],
        'transmission_type': [transmission_type],
        'age': [age],
        'accident': [1 if accident else 0],
        'clean_title': [1 if clean_title else 0]
    }

    return pd.DataFrame(input_data)

# Ana uygulama
def main():
    print("\n" + "="*50)
    print("Araç Fiyat Tahmini Uygulamasına Hoş Geldiniz!")
    print("="*50)

    # Modeli yükle
    model = get_model()

    while True:
        # Kullanıcı girişi al
        user_data = get_user_input()

        # Tahmin yap
        predicted_price = model.predict(user_data)[0]

        # Sonucu göster
        print("\n" + "="*50)
        print("TAHMİN SONUÇLARI")
        print("="*50)
        print(f"\nGirilen Araç Özellikleri:")
        print(f"- Marka: {user_data['brand'][0]}")
        print(f"- Model Yılı: {user_data['model_year'][0]} (Yaş: {user_data['age'][0]} yıl)")
        print(f"- Kilometre: {user_data['milage'][0]:,} km")
        print(f"- Yakıt Türü: {user_data['fuel_type'][0]}")
        print(f"- Motor Gücü: {user_data['engine_power'][0]} HP")
        print(f"- Motor Hacmi: {user_data['engine_volume'][0]} L")
        print(f"- Silindir Sayısı: {user_data['cylinders'][0]}")
        print(f"- Şanzıman Türü: {user_data['transmission_type'][0]}")
        print(f"- Kaza Geçmişi: {'Evet' if user_data['accident'][0] else 'Hayır'}")
        print(f"- Temiz Başlık: {'Evet' if user_data['clean_title'][0] else 'Hayır'}")

        print("\n" + "-"*50)
        print(f"\nTahmini Araç Fiyatı: ${predicted_price:,.2f}")
        print("-"*50)

        # Devam etmek isteyip istemediğini sor
        another = input("\nBaşka bir tahmin yapmak ister misiniz? (Evet/Hayır): ").lower()
        if another != 'evet':
            print("\nProgram sonlandırılıyor...")
            print("Görselleştirme dosyalarını kontrol etmeyi unutmayın!")
            break

if __name__ == "__main__":
    main()



Araç Fiyat Tahmini Uygulamasına Hoş Geldiniz!

Yeni model eğitiliyor...
Veri seti yükleniyor ve işleniyor...

Veri Seti Önizleme:
   id          brand              model  model_year  milage      fuel_type  \
0   0           MINI      Cooper S Base        2007  213000       Gasoline   
1   1        Lincoln              LS V8        2002  143250       Gasoline   
2   2      Chevrolet  Silverado 2500 LT        2002  136731  E85 Flex Fuel   
3   3        Genesis   G90 5.0 Ultimate        2017   19500       Gasoline   
4   4  Mercedes-Benz        Metris Base        2021    7388       Gasoline   

                                              engine  \
0       172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel   
1       252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel   
2  320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...   
3       420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel   
4       208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel   

                     transmission ext_col int_col  \
0         

  0%|          | 0/42 [00:00<?, ?it/s]